# Tutorial: Illumination Function Definition

This notebook isolates the incident wavefield used for propagation.

The illumination has two pieces:

- a scalar complex field, containing intensity and phase across the sample plane;
- a Jones-vector field, generated from the scalar field and the chosen polarization (`CR`, `CL`, `x`, or `y`).


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Make the notebook runnable from a fresh clone without requiring an editable install.
repo_root = Path.cwd()
if (repo_root / "src").exists() and str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

try:
    # Use the interactive widget backend when it is available in JupyterLab.
    %matplotlib widget
except Exception:
    # Plain scripts and some notebook renderers do not understand IPython magics.
    pass

plt.rcParams["figure.constrained_layout.use"] = True

from scattering_calculator.simulation_pipelines import simulation_configuration as sim
from scattering_calculator.sample_generator import pattern_generator


## 1. Define the sample-plane grid and X-ray beam

In [ ]:
shape = (512, 512)
real_space_pixel_size = 3e-9

xray_config = sim.XRayConfig(
    energy=778.0,
    photon_flux=1e10,
    pol="CR",
    coherence_length=(10e-6, 10e-6),
)


## 2. Gaussian illumination

`center` is a physical offset from the sample centre in metres. `fwhm` is the full width at half maximum of the beam waist. `distance` shifts the sample plane away from the beam waist and changes the wavefront curvature.


In [ ]:
illumination_config = sim.IlluminationConfig(
    XRayConfig=xray_config,
    shape=shape,
    real_space_pixel_size=real_space_pixel_size,
    illumination_function="gaussian",
    illumination_config={
        "center": np.array([0.0, 0.0]),   # y, x in m
        "distance": 1e-3,                 # m from waist
        "fwhm": 0.45e-6,                  # m at waist
    },
)
illumination_config.setup()
illumination_config.visualize_illumination()


## 3. Quantify beam size on the grid

This cell measures a simple line-profile FWHM after the field has been scaled to the requested photon flux.


In [ ]:
field = illumination_config.illumination.illumination
intensity = np.abs(field) ** 2
row = shape[0] // 2
x_um = illumination_config.illumination.x[row] * 1e6
profile = intensity[row] / intensity[row].max()

above_half = np.where(profile >= 0.5)[0]
measured_fwhm_um = (x_um[above_half[-1]] - x_um[above_half[0]]) if len(above_half) else np.nan

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(x_um, profile)
ax.axhline(0.5, color="k", ls="--", lw=1)
ax.set_xlabel("x in um")
ax.set_ylabel("normalized intensity")
ax.set_title(f"Central line profile, measured FWHM about {measured_fwhm_um:.3f} um")


## 4. Move the beam centre

This is useful for demonstrating partial illumination of an object hole or reference hole.


In [ ]:
centres = [
    np.array([0.0, 0.0]),
    np.array([0.20e-6, 0.0]),
    np.array([0.0, -0.20e-6]),
]

fig, axes = plt.subplots(1, len(centres), figsize=(11, 3.2), sharex=True, sharey=True)
for ax, centre in zip(axes, centres):
    illumination_config.update_illumination_config({
        "center": centre,
        "distance": 1e-3,
        "fwhm": 0.45e-6,
    })
    image = np.abs(illumination_config.illumination.illumination) ** 2
    ax.imshow(image, extent=1e6 * illumination_config.illumination.extent_real)
    ax.set_title(f"center = {centre * 1e6} um")
    ax.set_xlabel("x in um")
    ax.set_ylabel("y in um")


## 5. Compare Gaussian and plane-wave illumination

In [ ]:
gaussian_cfg = sim.IlluminationConfig(
    XRayConfig=sim.XRayConfig(778.0, 1e10, pol="CR"),
    shape=shape,
    real_space_pixel_size=real_space_pixel_size,
    illumination_function="gaussian",
    illumination_config={"center": np.array([0.0, 0.0]), "distance": 1e-3, "fwhm": 0.45e-6},
)
gaussian_cfg.setup()

plane_cfg = sim.IlluminationConfig(
    XRayConfig=sim.XRayConfig(778.0, 1e10, pol="CR"),
    shape=shape,
    real_space_pixel_size=real_space_pixel_size,
    illumination_function="plane_wave",
    illumination_config={},
)
plane_cfg.setup()

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), sharex=True, sharey=True)
for ax, cfg, title in zip(axes, [gaussian_cfg, plane_cfg], ["Gaussian", "Plane wave"]):
    ax.imshow(np.abs(cfg.illumination.illumination) ** 2, extent=1e6 * cfg.illumination.extent_real)
    ax.set_title(title)
    ax.set_xlabel("x in um")
    ax.set_ylabel("y in um")


## 6. Switch polarization without rebuilding the scalar beam

In [ ]:
for pol in ["CR", "CL", "x", "y"]:
    gaussian_cfg.update_polarization(pol)
    jones = gaussian_cfg.illumination.illumination_jones
    print(pol, "Jones field shape:", jones.shape, "component powers:", np.sum(np.abs(jones[..., 0])**2), np.sum(np.abs(jones[..., 1])**2))
